In [ ]:
# ==========================================
# BAGIAN 1: OPTUNA HYPERPARAMETER TUNING
# ==========================================

!pip install -q roboflow
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 11.2 MB/s eta 0:00:00


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="VI1yJoR4NJakTjkxhRlL")
project = rf.workspace("anno-n4jk2").project("penyakit_daun-xdq0c")
version = project.version(3)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Penyakit_Daun-3 in folder:: 100%|██████████| 5013/5013 [00:01<00:00, 4726.27it/s]


In [ ]:
# ==========================================
# IMPORT LIBRARY
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import numpy as np
import random
import os
import optuna
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
import json
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# PATH & KELAS
# ==========================================
data_dir = 'Penyakit_Daun-3'
train_dir = os.path.join(data_dir, 'train')
valid_dir = os.path.join(data_dir, 'valid')
test_dir = os.path.join(data_dir, 'test')

class_names = ["Bercak Daun", "Bulai", "Daun Sehat", "Hawar Daun", "Karat Daun"]
num_classes = len(class_names)
print(class_names)

['Bercak Daun', 'Bulai', 'Daun Sehat', 'Hawar Daun', 'Karat Daun']


In [ ]:
import os

def count_total_images(directory):
    total = 0
    for root, dirs, files in os.walk(directory):
        total += len([f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'))])
    return total

# Hitung total masing-masing split
total_train = count_total_images(train_dir)
total_valid = count_total_images(valid_dir)
total_test  = count_total_images(test_dir)

# Tampilkan hasil
print("="*40)
print(f"Total gambar di folder TRAIN  : {total_train}")
print(f"Total gambar di folder VALID  : {total_valid}")
print(f"Total gambar di folder TEST   : {total_test}")
print(f"Total keseluruhan             : {total_train + total_valid + total_test}")
print("="*40)

Total gambar di folder TRAIN  : 3494
Total gambar di folder VALID  : 750
Total gambar di folder TEST   : 749
Total keseluruhan             : 4993


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# ==========================================
# TRANSFORM SUPER MINIMAL & 100% DETERMINISTIK
# ==========================================
# Tidak ada augmentasi tambahan karena sudah dilakukan di Roboflow
minimal_transform = transforms.Compose([
    transforms.ToTensor(),  # WAJIB: JPG → Tensor + scale 0-1
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )                       # WAJIB untuk model pretrained ImageNet
])

# Training, validation, dan test PAKAI TRANSFORM YANG SAMA PERSIS
train_transform     = minimal_transform
valid_test_transform = minimal_transform

In [ ]:
# ==========================================
# SAVE ROOT & STUDY PATH
# ==========================================
SAVE_ROOT = "/content/drive/MyDrive/saved_models_with_optuna1"
os.makedirs(SAVE_ROOT, exist_ok=True)

DB_PATH = "/content/drive/MyDrive/optuna_study_penyakit_daun_jagung_terbaru.db"
storage = f"sqlite:///{DB_PATH}"

# Hapus DB lama jika ada
if os.path.exists(DB_PATH):
    print(f"Menghapus database Optuna lama: {DB_PATH}")
    os.remove(DB_PATH)

Menghapus database Optuna lama: /content/drive/MyDrive/optuna_study_penyakit_daun_jagung_terbaru.db


In [ ]:
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, loss):
        # Loss lebih bagus (lebih kecil)
        if loss < self.best_loss - self.min_delta:
            self.best_loss = loss
            self.counter = 0
        else:
            self.counter += 1

        # Patience habis → stop
        if self.counter >= self.patience:
            self.should_stop = True

In [ ]:
def train_and_evaluate(trial, params, plot_metrics=False, fine_tune_layers=0):
    seed_everything(42)

    # Dataset & Loader (sama seperti sebelumnya)
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_test_transform)
    test_dataset  = datasets.ImageFolder(test_dir,  transform=valid_test_transform)

    if len(train_dataset) == 0 or len(valid_dataset) == 0:
        raise ValueError("Dataset kosong! Periksa struktur folder.")

    train_loader = DataLoader(train_dataset, batch_size=params["batch"], shuffle=True,  num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=params["batch"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=params["batch"], shuffle=False, num_workers=2, pin_memory=True)

    # Model setup (sama)
    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    for param in model.parameters():
        param.requires_grad = False

    if fine_tune_layers > 0:
        num_features = len(model.features)
        for i in range(num_features - fine_tune_layers, num_features):
            for param in model.features[i].parameters():
                param.requires_grad = True
    else:
        for param in model.features.parameters():
            param.requires_grad = not params["freeze"]

    in_features = model.classifier[3].in_features
    model.classifier = nn.Sequential(
        model.classifier[0],
        model.classifier[1],
        nn.Dropout(p=params["dropout"]),
        nn.Linear(in_features, num_classes)
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = {
        "Adam": optim.Adam,
        "AdamW": optim.AdamW,
        "SGD": lambda p: optim.SGD(p, lr=params["lr"], momentum=0.9, weight_decay=1e-4)
    }[params["optimizer"]](trainable_params, lr=params["lr"])

    early_stopper = EarlyStopping(patience=15, min_delta=0.001)

    # === FOLDER PENYIMPANAN HASIL ===
    if trial is not None:
        trial_dir = os.path.join(SAVE_ROOT, f"trial_{trial.number}")
    else:
        trial_dir = os.path.join(SAVE_ROOT, "BEST_RETRAIN_FINAL")
    os.makedirs(trial_dir, exist_ok=True)

    best_model_path = os.path.join(trial_dir, "best_model.pth")

    # Metrics storage
    history = {
        "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
        "precision": [], "recall": [], "f1": []
    }

    best_valid_acc = 0.0

    print(f"\n{'='*80}")
    print(f"TRAINING {'TRIAL ' + str(trial.number) if trial else 'BEST MODEL (RETRAIN)'}")
    print(f"Folder: {trial_dir}")
    print(f"Params → Batch: {params['batch']} | LR: {params['lr']:.2e} | Opt: {params['optimizer']} | "
          f"Freeze: {params['freeze']} | Dropout: {params['dropout']} | Fine-tune layers: {fine_tune_layers}")
    print(f"{'='*80}\n")

    epochs = params["epochs"] if fine_tune_layers == 0 else 50

    for epoch in range(epochs):
        # TRAIN
        model.train()
        train_loss, correct, total = 0.0, 0, 0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()

        train_loss /= total
        train_acc = correct / total

        # VALIDATION
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, pred = outputs.max(1)
                total += labels.size(0)
                correct += pred.eq(labels).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= total
        val_acc = correct / total
        prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)

        # Simpan ke history
        for k, v in zip(["train_loss","val_loss","train_acc","val_acc","precision","recall","f1"],
                        [train_loss, val_loss, train_acc, val_acc, prec, rec, f1]):
            history[k].append(v)

        # Report ke Optuna
        if trial:
            trial.report(val_acc, epoch)

        # Save best model
        if val_acc > best_valid_acc:
            best_valid_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  [BEST] Epoch {epoch+1} → Val Acc: {val_acc*100:6.2f}% ↑ (saved)")

        # Early stopping
        if early_stopper.step(val_loss):
            print(f"  [EARLY STOP] Epoch {epoch+1}")
            break

        print(f"  Epoch {epoch+1:2d} | Train Acc: {train_acc*100:6.2f}% | Val Acc: {val_acc*100:6.2f}% | "
              f"Prec: {prec:.3f} | Rec: {rec:.3f} | F1: {f1:.3f}")

    # ===================================
    # FINAL TEST + SAVE ALL ARTIFACTS
    # ===================================
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    test_acc = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=train_dataset.classes, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n[FINAL RESULT] Test Accuracy: {test_acc*100:.2f}% | Best Val Acc: {best_valid_acc*100:.2f}%\n")

    # === SIMPAN SEMUA KE DRIVE ===
    # 1. Model
    final_model_path = os.path.join(trial_dir, "final_model.pth")
    torch.save(model.state_dict(), final_model_path)

    # 2. History (untuk di-load lagi nanti)
    with open(os.path.join(trial_dir, "history.json"), "w") as f:
        json.dump({k: [float(x) for x in v] for k, v in history.items()}, f, indent=2)

    # 3. Classification Report
    with open(os.path.join(trial_dir, "classification_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    # 4. Grafik-grafik (selalu disimpan saat retrain terbaik)
    if len(history["train_loss"]) > 0:
        epochs_range = range(1, len(history["train_loss"]) + 1)

        # Loss
        plt.figure(figsize=(10,6))
        plt.plot(epochs_range, history["train_loss"], label="Train Loss", marker='o')
        plt.plot(epochs_range, history["val_loss"], label="Val Loss", marker='o')
        plt.title("Training and Validation Loss")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
        plt.savefig(os.path.join(trial_dir, "loss_curve.png"), dpi=300, bbox_inches='tight')
        plt.close()

        # Accuracy
        plt.figure(figsize=(10,6))
        plt.plot(epochs_range, history["train_acc"], label="Train Acc", marker='o')
        plt.plot(epochs_range, history["val_acc"], label="Val Acc", marker='o')
        plt.title("Training and Validation Accuracy")
        plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(True)
        plt.savefig(os.path.join(trial_dir, "accuracy_curve.png"), dpi=300, bbox_inches='tight')
        plt.close()

        # Precision, Recall, F1
        plt.figure(figsize=(10,6))
        plt.plot(epochs_range, history["precision"], label="Precision", marker='o')
        plt.plot(epochs_range, history["recall"], label="Recall", marker='o')
        plt.plot(epochs_range, history["f1"], label="F1-Score", marker='o')
        plt.title("Precision, Recall, F1-Score (Validation)")
        plt.xlabel("Epoch"); plt.ylabel("Score"); plt.legend(); plt.grid(True)
        plt.savefig(os.path.join(trial_dir, "metrics_curve.png"), dpi=300, bbox_inches='tight')
        plt.close()

        # Confusion Matrix
        plt.figure(figsize=(8,7))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=train_dataset.classes, yticklabels=train_dataset.classes)
        plt.title("Confusion Matrix - Test Set")
        plt.ylabel("True Label"); plt.xlabel("Predicted Label")
        plt.savefig(os.path.join(trial_dir, "confusion_matrix.png"), dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Semua grafik, model, report, dan confusion matrix telah disimpan di:")
        print(f"   → {trial_dir}")

    return {
        "test_acc": float(test_acc),
        "best_val_acc": float(best_valid_acc)
    }

In [ ]:
def objective(trial):

    params = {
        "epochs": 100,
        "batch": trial.suggest_categorical("batch", [16, 32, 64]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-1, log=True),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "freeze": trial.suggest_categorical("freeze", [True]),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
    }

    # --- jalankan training ---
    result = train_and_evaluate(trial, params)

    # --- simpan ke dashboard ---
    trial.set_user_attr("best_val_acc", result["best_val_acc"])
    trial.set_user_attr("test_acc", result["test_acc"])

    return result["best_val_acc"]  # Maximize berdasarkan val acc

In [ ]:
# ==========================================
# JALANKAN OPTUNA — TANPA PRUNING, OUTPUT SESUAI CONTOH
# ==========================================

# BUAT STUDY BARU — TANPA PRUNER
study = optuna.create_study(
    direction="maximize",
    study_name="penyakit_daun_tuning_v1",
    storage=storage,
    load_if_exists=True
)
print(f"\nMulai optimasi")
print(f"Database: {DB_PATH}\n")

study.optimize(objective, n_trials=40)

In [ ]:
# Tampilkan hyperparameter importance di dashboard Optuna
import optuna.visualization as vis
vis.plot_param_importances(study)

In [ ]:
# ==========================================
# RETRAIN DAN TEST ULANG BEST TRIAL BERDASARKAN VAL ACC
# ==========================================

# Dapatkan best trial berdasarkan val acc
best_trial = study.best_trial

best_params = {
    "epochs": 100,
    "batch": best_trial.params["batch"],
    "lr": best_trial.params["lr"],
    "optimizer": best_trial.params["optimizer"],
    "freeze": best_trial.params["freeze"],
    "dropout": best_trial.params["dropout"]
}

print(f"BEST TRIAL: {best_trial.number} | Val Acc: {best_trial.user_attrs.get('best_val_acc', best_trial.value)*100:.2f}%")

result = train_and_evaluate(None, best_params, plot_metrics=True)

BEST TRIAL: 4 | Val Acc: 99.87%

TRAINING BEST MODEL (RETRAIN)
Folder: /content/drive/MyDrive/saved_models_with_optuna1/BEST_RETRAIN_FINAL
Params → Batch: 16 | LR: 4.21e-03 | Opt: AdamW | Freeze: True | Dropout: 0.5 | Fine-tune layers: 0



  [BEST] Epoch 1 → Val Acc:  98.53% ↑ (saved)
  Epoch  1 | Train Acc:  93.76% | Val Acc:  98.53% | Prec: 0.985 | Rec: 0.985 | F1: 0.985


  [BEST] Epoch 2 → Val Acc:  99.20% ↑ (saved)
  Epoch  2 | Train Acc:  96.85% | Val Acc:  99.20% | Prec: 0.992 | Rec: 0.992 | F1: 0.992


  [BEST] Epoch 3 → Val Acc:  99.47% ↑ (saved)
  Epoch  3 | Train Acc:  97.25% | Val Acc:  99.47% | Prec: 0.994 | Rec: 0.995 | F1: 0.995


  Epoch  4 | Train Acc:  97.45% | Val Acc:  98.93% | Prec: 0.989 | Rec: 0.989 | F1: 0.989


...

  Epoch 96 | Train Acc:  97.82% | Val Acc:  99.07% | Prec: 0.990 | Rec: 0.991 | F1: 0.991


  Epoch 97 | Train Acc:  97.60% | Val Acc:  99.20% | Prec: 0.992 | Rec: 0.992 | F1: 0.992


  Epoch 98 | Train Acc:  97.68% | Val Acc:  99.47% | Prec: 0.994 | Rec: 0.995 | F1: 0.995


  Epoch 99 | Train Acc:  97.77% | Val Acc:  99.07% | Prec: 0.990 | Rec: 0.991 | F1: 0.990


  Epoch 100 | Train Acc:  97.51% | Val Acc:  99.20% | Prec: 0.992 | Rec: 0.992 | F1: 0.992

[FINAL RESULT] Test Accuracy: 98.66% | Best Val Acc: 99.87%

Semua grafik, model, report, dan confusion matrix telah disimpan di:
   → /content/drive/MyDrive/saved_models_with_optuna1/BEST_RETRAIN_FINAL
